## FOV and Timepoint Normalization

- consider adding detrending within FOV using axis projected data preinduction
- this would likely bring average sizes down even further
- if I want to do this, do it before pooling data!!!
- should at least check the extent of the problem before moving too far

In [ ]:
import paulssonlab.deaton.trenchripper.trenchripper as tr

import numpy as np
import scipy as sp
import dask.dataframe as dd
import pandas as pd
from dask.distributed import wait

from matplotlib import pyplot as plt
import holoviews as hv
hv.extension('bokeh')

In [ ]:
# addition of active memory manager
import dask
dask.config.set({'distributed.scheduler.active-memory-manager.start': True});
dask.config.set({'distributed.scheduler.worker-ttl': "5m"});
dask.config.set({'distributed.scheduler.allowed-failures': 100});

### Functions for this notebook

In [ ]:
def get_aligned_loc_from_index(df,idx_series):
    df_out = df.loc[idx_series.tolist()]
    return df_out

def index_loc_lookup(df,idx_series):
    df_out = dd.map_partitions(get_aligned_loc_from_index,df,idx_series,align_dataframes=False)
    return df_out

def hrm_find_mode(series, max_iter=1000, min_binsize=50):
    working_series = series
    for i in range(max_iter):
        range_max,range_min = np.max(working_series),np.min(working_series)
        midpoint = (range_max+range_min)/2
        above_middle = working_series[working_series>midpoint]
        below_middle = working_series[working_series<=midpoint]
        
        count_above = len(above_middle)
        count_below = len(below_middle)
        
        
        if count_above>count_below:
            working_series = above_middle
        else:
            working_series = below_middle
        
        if i > 0:
            if (len(working_series)<min_binsize) or (last_midpoint==midpoint):
                return np.mean(working_series)

        last_midpoint = midpoint
        
def bootstrap_hrm(series, n_bootstraps=100, max_n_per_bootstrap=100):
    modes = []
    
    series_len = len(series)
    
    n_per_bootstrap = min(series_len,max_n_per_bootstrap)

    for n in range(n_bootstraps):
        modes.append(hrm_find_mode(series.sample(n=n_per_bootstrap)))
    return np.mean(modes)

def get_normal_fovs(fov_series, med_filter_size=5, n_stds=2):
    median_series = sp.ndimage.median_filter(fov_series,size=(med_filter_size,),mode="mirror")
    
    residuals = fov_series-median_series

    gaussian_fit = sp.stats.norm.fit(residuals)
    gaussian_fit = sp.stats.norm(loc=gaussian_fit[0], scale=gaussian_fit[1])

    lower,upper = (-n_stds*gaussian_fit.std(),n_stds*gaussian_fit.std())
        
    thr_mask = ((residuals>lower)&(residuals<upper))
    
    return thr_mask

#### Start Dask

In [ ]:
dask_controller = tr.trcluster.dask_controller(
    walltime="2:00:00",
    local=False,
    n_workers=700,
    n_workers_min=100,
    memory='8GB',#"16GB",
    working_directory="/home/lag36/scratch/lag36/dask/",
)
dask_controller.startdask()

In [ ]:
dask_controller.displaydashboard()

### Import Lineage dfs

In [ ]:
lineage_df_paths = ["/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/Growth_Division/lineage/withNucleoidData/"]
lineage_df_list = [dd.read_parquet(path,calculate_divisions=True) for path in lineage_df_paths]

# #hack assuming both datasets are the same size, within a factor of 2
max_cellid = lineage_df_list[0]['CellID'].max().compute()
max_timepoint = lineage_df_list[0]['timepoints'].max().compute()
max_file_trench_idx = lineage_df_list[0]['File Trench Index'].max().compute()
max_file_idx = lineage_df_list[0]['File Index'].max().compute()

cellid_digits = len(str(max_cellid*2))
timepoint_digits = len(str(max_timepoint))
file_trench_idx_digits = len(str(max_file_trench_idx*2))
file_idx_digits = len(str(max_file_idx*2))
print(cellid_digits,timepoint_digits,file_trench_idx_digits,file_idx_digits)

for i in range(len(lineage_df_list)):
    lineage_df_list[i]["Experiment #"] = i
    lineage_df_list[i] = lineage_df_list[i].reset_index()
    
    parq_filemultiexp_idx = lineage_df_list[i].apply(lambda x: int(f'{int(x["Experiment #"]):02n}{int(x["File Index"]):0{file_idx_digits}n}{int(x["File Trench Index"]):0{file_trench_idx_digits}n}{int(x["timepoints"]):0{timepoint_digits}n}{int(x["CellID"]):0{cellid_digits}n}'),\
                                                     axis=1, meta=("Multi-Experiment File Parquet Index",int))
        
    lineage_df_list[i]["Multi-Experiment File Parquet Index"] = parq_filemultiexp_idx.persist()
    lineage_df_list[i] = lineage_df_list[i].set_index("Multi-Experiment File Parquet Index",sorted=True)

parq_filemultiexp_idx = lineage_df_list[i].apply(lambda x: int(f'{int(x["Experiment #"]):02n}{int(x["File Index"]):08n}{int(x["File Trench Index"]):04n}{int(x["timepoints"]):04n}{int(x["CellID"]):04n}'), axis=1)
    
lineage_df_merged = dd.concat(lineage_df_list)

lineage_df_merged["Multi-Experiment trenchid"] = lineage_df_merged.apply(lambda x: int(f'{int(x["Experiment #"]):02n}{int(x["trenchid"]):08n}'), axis=1, meta=("Multi-Experiment trenchid", int))
lineage_df_merged["Multi-Experiment timepoints"] = lineage_df_merged.apply(lambda x: int(f'{int(x["Experiment #"]):02n}{int(x["timepoints"]):08n}'), axis=1, meta=("Multi-Experiment timepoints", int))
lineage_df_merged["Multi-Experiment fov"] = lineage_df_merged.apply(lambda x: int(f'{int(x["Experiment #"]):02n}{int(x["fov"]):08n}'), axis=1, meta=("Multi-Experiment fov", int))

dd.to_parquet(lineage_df_merged, "/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/Growth_Division/2023-01-24_Merged_Lineage",engine='pyarrow',compression='gzip',write_metadata_file=True,overwrite=True)

In [ ]:
lineage_df = dd.read_parquet("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/Growth_Division/2023-01-24_Merged_Lineage", calculate_divisions=True)

In [ ]:
# Remove early anucleated cells
crispr_induction_timepoint = 13 # 41

lineage_df = lineage_df.persist()
lineage_df_early = lineage_df[lineage_df["timepoints"] < 13].persist()
lineage_df_early = lineage_df_early.reset_index(drop = False).set_index("Multi-Experiment trenchid",sorted=True).persist()
mean_tot_nuc_area = lineage_df_early["Total Nucleoid Area"].groupby("Multi-Experiment trenchid").apply(lambda x: np.mean(x),meta=("Total Nucleoid Area",float)).compute()
bad_trenches = mean_tot_nuc_area[mean_tot_nuc_area == 0].index.values
lineage_df_filt = lineage_df[~lineage_df["Multi-Experiment trenchid"].isin(bad_trenches)]

dd.to_parquet(lineage_df_filt, "/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/Growth_Division/2023-01-24_Merged_Lineage_RemovedAnucleatedCells",engine='pyarrow',compression='gzip',write_metadata_file=True,overwrite=True)

In [ ]:
lineage_df = dd.read_parquet("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/Growth_Division/2023-01-24_Merged_Lineage_RemovedAnucleatedCells", calculate_divisions=True)

In [ ]:
# ##temp fix
# lineage_df["CellID"] = lineage_df["CellID"].astype(int)
# lineage_df["Global CellID"] = lineage_df["Global CellID"].astype(int)

### Scaling to uninduced values
 - Comparing measured distribution to the dataset-wide distribution to rescale (measuring both mean and variance)
     - actually got rid of the variance contribution to the scaling, since it is a less reliable estimator
     - especially falls apart when the lineage gets a little heterogenous
 - Could be useful for aligning datasets later
 - Also consider having the derivative values (volume, surface area) recalculated based on the adjustment

In [ ]:
# crispr_induction_timepoint = 13 #41
sample_rate_for_plots = 0.02

values_to_rescale = ['YFP mean_intensity','area', 'major_axis_length', 'minor_axis_length',"Volume","Surface Area", 'mCherry mean_intensity',
                     "Total Nucleoid Area", 'Nucleoid Area Mean', 'Nucleoid Mean Intensity Mean','Nucleoid Major Axis Length Mean', 'Nucleoid Minor Axis Length Mean'] ### INCLUDED TOTAL NUCLEOID AREA
subsetted_lineage_df = lineage_df[["fov","timepoints","Cell Y (local)","Cell X (local)","trenchid","Multi-Experiment trenchid","Multi-Experiment timepoints","Multi-Experiment fov","Experiment #"] + values_to_rescale]
subsetted_lineage_df_early = subsetted_lineage_df[subsetted_lineage_df["timepoints"]<crispr_induction_timepoint]

subsetted_lineage_df_trenchid_sorted = subsetted_lineage_df.reset_index(drop=False).set_index("Multi-Experiment trenchid").persist()
# subsetted_lineage_df_trenchid_sorted_reset_idx = subsetted_lineage_df_trenchid_sorted.reset_index(drop=False).persist()
subsetted_lineage_df_early_trenchid_sorted = subsetted_lineage_df_early.set_index("Multi-Experiment trenchid")

subsetted_lineage_df_early_subsampled = subsetted_lineage_df_early.sample(frac=sample_rate_for_plots).compute()
subsetted_lineage_df_early_subsampled_trenchid_sorted = subsetted_lineage_df_early_subsampled.set_index("Multi-Experiment trenchid").sort_index()
# subsetted_lineage_df_early_subsampled_trenchid_sorted_reset_idx = subsetted_lineage_df_early_subsampled_trenchid_sorted.reset_index(drop=False)

In [ ]:
fig = plt.figure(figsize=(30,20))

for i,label in enumerate(values_to_rescale):
    trench_means = subsetted_lineage_df_early_trenchid_sorted[label].groupby("Multi-Experiment trenchid").apply(lambda x: np.mean(x),meta=(label,float)).compute()
    global_median_of_means = np.median(trench_means)
    rescaling_lookup = trench_means/global_median_of_means
    
    rescaled_subsetted_lineage_df_trenchid_sorted = (subsetted_lineage_df_trenchid_sorted[label]/rescaling_lookup).dropna()
    rescaled_subsetted_lineage_df_early_subsampled_trenchid_sorted = (subsetted_lineage_df_early_subsampled_trenchid_sorted[label]/rescaling_lookup).dropna()
    
    subsetted_lineage_df_trenchid_sorted["Rescaled: " + label] = rescaled_subsetted_lineage_df_trenchid_sorted.persist()#.reset_index(drop=True).persist()
    wait(subsetted_lineage_df_trenchid_sorted["Rescaled: " + label]);
    
    subsetted_lineage_df_early_subsampled_trenchid_sorted["Rescaled: " + label] = rescaled_subsetted_lineage_df_early_subsampled_trenchid_sorted#.reset_index(drop=True)
    
    experiment_list = list(subsetted_lineage_df_early_subsampled_trenchid_sorted["Experiment #"].unique())
    
    plt.subplot(3,4,i+1) # TODO Change according to len(values_to_rescale)
    for exp_num in experiment_list:
        exp_subset = subsetted_lineage_df_early_subsampled_trenchid_sorted[subsetted_lineage_df_early_subsampled_trenchid_sorted["Experiment #"] == exp_num]
        plt.hist(exp_subset[label],
             range=(0,np.percentile(subsetted_lineage_df_early_subsampled_trenchid_sorted[label],99)),
             bins=50,alpha=1,histtype='step',label="Experiment " + str(exp_num), density=True)
        
    plt.hist(subsetted_lineage_df_early_subsampled_trenchid_sorted[label],
             range=(0,np.percentile(subsetted_lineage_df_early_subsampled_trenchid_sorted[label],99)),
             bins=50,alpha=1,histtype='step',label="All Experiments", density=True)
    plt.hist(subsetted_lineage_df_early_subsampled_trenchid_sorted["Rescaled: " + label],
             range=(0,np.percentile(subsetted_lineage_df_early_subsampled_trenchid_sorted[label],99)),
             bins=50,alpha=1,histtype='step',label="All Experiments Rescaled", density=True)
    plt.ylabel("# Cells", fontsize=18)
    plt.xlabel(values_to_rescale[i], fontsize=18)
    plt.xticks(fontsize=18)
    plt.yticks(fontsize=18)
    plt.legend()
                      
plt.savefig("Trench_correction_replicate_1.png",dpi=500)

In [ ]:
# ## OLD VERSION FOR DEALING WITH ANUCLEATED CELLS


# fig = plt.figure(figsize=(30,20))

# for i,label in enumerate(values_to_rescale):
#     trench_means = subsetted_lineage_df_early_trenchid_sorted[label].groupby("Multi-Experiment trenchid").apply(lambda x: np.mean(x),meta=(label,float)).compute()
#     global_median_of_means = np.median(trench_means)
#     rescaling_lookup = trench_means/global_median_of_means
        
#     if label in ['Total Nucleoid Area', 'Number of Nucleoids']:
#         rescaling_lookup[rescaling_lookup == 0] = -1
#         temp = (subsetted_lineage_df_trenchid_sorted[label]/rescaling_lookup).dropna()
#         temp = temp.where( temp >=0, 0 )
#         rescaled_subsetted_lineage_df_trenchid_sorted = temp
        
#         temp = (subsetted_lineage_df_early_subsampled_trenchid_sorted[label]/rescaling_lookup).dropna()
#         temp[temp < 0] = 0
#         rescaled_subsetted_lineage_df_early_subsampled_trenchid_sorted = temp
#     else:
#         rescaled_subsetted_lineage_df_trenchid_sorted = (subsetted_lineage_df_trenchid_sorted[label]/rescaling_lookup).dropna()
#         rescaled_subsetted_lineage_df_early_subsampled_trenchid_sorted = (subsetted_lineage_df_early_subsampled_trenchid_sorted[label]/rescaling_lookup).dropna()
            
#     subsetted_lineage_df_trenchid_sorted["Rescaled: " + label] = rescaled_subsetted_lineage_df_trenchid_sorted.persist()#.reset_index(drop=True).persist()
#     wait(subsetted_lineage_df_trenchid_sorted["Rescaled: " + label]);
    
#     subsetted_lineage_df_early_subsampled_trenchid_sorted["Rescaled: " + label] = rescaled_subsetted_lineage_df_early_subsampled_trenchid_sorted#.reset_index(drop=True)
    
#     experiment_list = list(subsetted_lineage_df_early_subsampled_trenchid_sorted["Experiment #"].unique())
    
#     plt.subplot(2,4,i+1)
#     for exp_num in experiment_list:
#         exp_subset = subsetted_lineage_df_early_subsampled_trenchid_sorted[subsetted_lineage_df_early_subsampled_trenchid_sorted["Experiment #"] == exp_num]
#         plt.hist(exp_subset[label],
#              range=(0,np.percentile(subsetted_lineage_df_early_subsampled_trenchid_sorted[label],99)),
#              bins=50,alpha=1,histtype='step',label="Experiment " + str(exp_num), density=True)
        
#     plt.hist(subsetted_lineage_df_early_subsampled_trenchid_sorted[label],
#              range=(0,np.percentile(subsetted_lineage_df_early_subsampled_trenchid_sorted[label],99)),
#              bins=50,alpha=1,histtype='step',label="All Experiments", density=True)
#     plt.hist(subsetted_lineage_df_early_subsampled_trenchid_sorted["Rescaled: " + label],
#              range=(0,np.percentile(subsetted_lineage_df_early_subsampled_trenchid_sorted[label],99)),
#              bins=50,alpha=1,histtype='step',label="All Experiments Rescaled", density=True)
#     plt.ylabel("# Cells", fontsize=18)
#     plt.xlabel(values_to_rescale[i], fontsize=18)
#     plt.xticks(fontsize=18)
#     plt.yticks(fontsize=18)
#     plt.legend()
                      
# plt.savefig("Trench_correction_replicate_1.png",dpi=500)

### Variables over FOVs
- only eliminating fovs with outlier values

In [ ]:
crispr_induction_timepoint = 13

fov_sorted_lineage_df = subsetted_lineage_df_trenchid_sorted.reset_index(drop=False).set_index("Multi-Experiment fov")
fov_sorted_lineage_df = fov_sorted_lineage_df.drop(columns = values_to_rescale + ["Cell Y (local)","Cell X (local)"]\
                                                    ).rename(columns={"Rescaled: " + name:name for name in values_to_rescale}).persist()
fig = plt.figure(figsize=(30,20))
values_to_rescale = ['YFP mean_intensity','area', 'major_axis_length', 'minor_axis_length',"Volume", "Surface Area", 'mCherry mean_intensity',
                     "Total Nucleoid Area", 'Nucleoid Area Mean', 'Nucleoid Mean Intensity Mean','Nucleoid Major Axis Length Mean', 'Nucleoid Minor Axis Length Mean'] # NUCLEOID AREA
values_names = ['Mean YFP Intensity','Area', 'Major Axis Length', 'Minor Axis Length',"Volume","Surface Area", 'mCherry mean_intensity',
                "Total Nucleoid Area", 'Nucleoid Area Mean', 'Nucleoid Mean Intensity Mean','Nucleoid Major Axis Length Mean', 'Nucleoid Minor Axis Length Mean']
normal_fov_list = []
lineage_df_subsample = fov_sorted_lineage_df[fov_sorted_lineage_df["timepoints"]<crispr_induction_timepoint]

for i,label in enumerate(values_to_rescale):

    plt.subplot(3,4,i+1)
    
    plt.title(values_names[i],fontsize=22)
    plt.xlabel("FOV #", fontsize=18)
    plt.ylabel("Scaling", fontsize=18)
    plt.xticks(fontsize=18)
    plt.yticks(fontsize=18)
    plt.ylim(0.,2.)
    
    fov_series_groupby = lineage_df_subsample.groupby("Multi-Experiment fov",sort=False)[label]
    fov_median_series = fov_series_groupby.apply(lambda x: np.median(x), meta=float).compute().sort_index()

    normal_fov_series = get_normal_fovs(fov_median_series)
    fov_median_series = fov_median_series[normal_fov_series]
    normal_fovs = set(fov_median_series.index)
    normal_fov_list.append(normal_fovs)
    
    plt.plot((fov_median_series/np.median(fov_median_series)).values)
    ##maybe add a line to divide the experiments
    plt.legend()
filtered_fov_list = list(set.intersection(*normal_fov_list))
dask_controller.daskclient.cancel(fov_sorted_lineage_df)

lineage_df_fov_correction = lineage_df[["Multi-Experiment timepoints","Multi-Experiment fov"] + values_to_rescale]
lineage_df_fov_correction = lineage_df_fov_correction[lineage_df_fov_correction["Multi-Experiment fov"].isin(filtered_fov_list)].persist()

plt.savefig("FOV_check_replicate_1.png",dpi=500)

### Variables over time

In [ ]:
target_samples = 100000 

values_to_rescale_step_2 = ["YFP mean_intensity", 'mCherry mean_intensity', 'Nucleoid Mean Intensity Mean'] # Change this
normalization_fn = [np.max, np.max, np.max] #Change this
ttl_samples = len(lineage_df_fov_correction)
frac_to_sample = target_samples/ttl_samples
lineage_df_subsample = lineage_df_fov_correction.sample(frac=frac_to_sample).compute()

In [ ]:
fig = plt.figure(figsize=(30,20))
values_names = ['Mean YFP Intensity', 'Mean mCherry Intensity', 'Nucleoid Mean Intensity Mean']
for i,label in enumerate(values_to_rescale_step_2):
    time_series_groupby = lineage_df_subsample.groupby("Multi-Experiment timepoints")[label]
    time_mode_series = time_series_groupby.apply(lambda x: bootstrap_hrm(x)).sort_index()
    time_correction_series = time_mode_series/normalization_fn[i](time_mode_series)
    time_correction_dict = time_correction_series.to_dict()
    plt.subplot(2,3,i+1)
    plt.plot(time_correction_series.values)
    plt.title(values_names[i],fontsize=22)
    plt.xlabel("Timepoint (10 min steps)", fontsize=18)
    #line seperating experiments
    plt.ylabel("Scaling", fontsize=18)
    plt.xticks(fontsize=18)
    plt.yticks(fontsize=18) 
    plt.ylim(0.,1.)
    lineage_df_fov_correction[label + ": Time Corrected"] = lineage_df_fov_correction.apply(lambda x: x[label]/time_correction_dict[x["Multi-Experiment timepoints"]], meta=float, axis=1).persist()
plt.savefig("Time_correction_replicate_1.png",dpi=500)

### Overwrite Variables with Correction
#### ADD HERE NEW COLUMNS TO CARRY OVER!

In [ ]:
values_to_rescale

In [ ]:
other_vars_to_carry = ['solidity', 'Nucleoid Separation Mean','Nucleoid Separation CV','Nucleoid Separation Min','Nucleoid Separation Max',
                         'Nucleoid Area CV','Nucleoid Solidity Mean','Nucleoid Solidity CV','Nucleoid Mean Intensity CV']

In [ ]:
corrected_lineage_df  = index_loc_lookup(lineage_df,lineage_df_fov_correction.index)

# only_fov_scaled_values = list(set(values_to_rescale)-set(values_to_rescale_step_2))
# for label in only_fov_scaled_values:
#     corrected_lineage_df[label] = lineage_df_fov_correction[label + ': FOV Corrected']
for label in values_to_rescale_step_2:
    corrected_lineage_df[label] = lineage_df_fov_correction[label + ': Time Corrected']

# ADD HERE NEW VARIABLES!!! NUCLEOIDS
corrected_lineage_df = corrected_lineage_df[['fov', 'row', 'trench', 'trenchid' , 'timepoints', 'Mother', 'File Index','File Trench Index','CellID',\
       'Global CellID', 'Trench Score', 'Mother CellID', 'Daughter CellID 1','Daughter CellID 2', 'Sister CellID',\
        'Centroid X', 'Centroid Y','Cell Y (local)','Cell X (local)','Kymograph File Parquet Index','Kymograph FOV Parquet Index','FOV Parquet Index',\
        'File Parquet Index','Experiment #', 'Segment Index', 'bbox_min_row', 'bbox_min_col', 'bbox_max_row', 'bbox_max_col', 'Trenchid Timepoint Index', 'Number of Nucleoids'] + values_to_rescale + other_vars_to_carry]

In [ ]:
lineage_df_paths = ["/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/Growth_Division/lineage/withNucleoidData"]
lineage_df_list = [dd.read_parquet(path,calculate_divisions=True) for path in lineage_df_paths]
output_path_list = ["/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/Growth_Division/2023-01-24_FOV_and_Time_Normalization"]

for i,df in enumerate(lineage_df_list):
    df_mask = (corrected_lineage_df['Experiment #'] == i).persist()
    corrected_lineage_df_masked = corrected_lineage_df[df_mask]
    corrected_lineage_df_masked = corrected_lineage_df_masked.set_index('File Parquet Index',sorted=True).repartition(divisions=df.divisions,force=True)
    corrected_lineage_df_masked.to_parquet(output_path_list[i], engine="pyarrow", overwrite=True)

In [ ]:
dask_controller.shutdown()

### Variables within FOV
 - Currently not doing this in favor of rescaling to the preinduced trench

In [ ]:
import sklearn as skl
import numpy as np
import pandas as pd
import scipy as sp

def detrend_FOV(fov_df,crispr_induction_timepoint = 24,percentile_list = [50,50,50,50,50,50],values_to_rescale = ['mCherry mean_intensity','area', 'major_axis_length', 'minor_axis_length',"Volume","Surface Area"]):
    if len(fov_df)>0:
        lineage_df_early_timepoints = fov_df[fov_df["timepoints"]<crispr_induction_timepoint]

        cell_positions = lineage_df_early_timepoints[["Cell y (local)","Cell x (local)"]].values
        obs_vals = lineage_df_early_timepoints[values_to_rescale].values
        poly = skl.preprocessing.PolynomialFeatures(degree=2)
        poly_transform = poly.fit_transform(cell_positions)
        reg = skl.linear_model.LinearRegression().fit(poly_transform, obs_vals)

        cell_positions_full = fov_df[["Cell y (local)","Cell x (local)"]].values
        obs_vals_full = fov_df[values_to_rescale].values
        poly_transform_full = poly.fit_transform(cell_positions_full)

        predicted_vals = reg.predict(poly_transform_full)
        residuals = obs_vals_full-predicted_vals

        percentile_baselined_vals = np.array([np.percentile(obs_vals[:,i],percentile_list[i]) for i in range(len(values_to_rescale))])
        percentile_detrended_data = residuals+percentile_baselined_vals

        percentile_detrended_data = pd.DataFrame({val + ": FOV Detrended":percentile_detrended_data[:,i] for i,val in enumerate(values_to_rescale)})
        fov_df = fov_df.reset_index()
        fov_df = fov_df.join(percentile_detrended_data).set_index("fov")
        return fov_df
    
    else:
        return fov_df

In [ ]:
values_to_rescale = ['mCherry mean_intensity','area', 'major_axis_length', 'minor_axis_length',"Volume","Surface Area"]
values_names = ['Mean mCherry Intensity','Area', 'Major Axis Length', 'Minor Axis Length',"Volume","Surface Area"]

fov_sorted_lineage_df = lineage_df[["fov","timepoints","Cell y (local)","Cell x (local)"] + values_to_rescale]
fov_sorted_lineage_df = fov_sorted_lineage_df.reset_index().set_index("fov",sorted=True)
n_fovs = fov_sorted_lineage_df.divisions[-1]+1

fov_sorted_lineage_df = fov_sorted_lineage_df.repartition(divisions=list(range(n_fovs))).persist()

fov_sorted_lineage_df_detrended = dd.map_partitions(detrend_FOV,fov_sorted_lineage_df)
fov_sorted_lineage_df_detrended = fov_sorted_lineage_df_detrended.drop(columns = values_to_rescale + ["Cell y (local)","Cell x (local)"]).rename(columns={name+": FOV Detrended":name for name in values_to_rescale}).persist()

In [ ]:
fov_sorted_lineage_df_detrended.head()